# Журнал изменений версий моделей в эксплуатации

Ноутбук формирует только один отчёт — `model_change_log.xlsx`.

Периметр: все актуальные версии моделей с `model_ver_prom_expl_flag = true`, без отбора по категории значимости. Каждая строка отчёта соответствует одному изменению параметра версии модели.


In [ ]:
import os
import sys

os.environ["SPARK_MAJOR_VERSION"] = "3.5.1"
os.environ["SPARK_HOME"] = "/usr/sdp/current/spark3.5.1-client/"
os.environ["PYSPARK_DRIVER"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/")
sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/lib/py4j-0.10.9.7-src.zip")

from pathlib import Path

from pyspark import SparkConf
from pyspark.sql import DataFrame, SparkSession, Window, functions as F

conf = (
    SparkConf()
    .setAppName("model_version_change_log")
    .setMaster("yarn")
    .set("spark.executor.cores", "2")
    .set("spark.executor.memory", "6g")
    .set("spark.executor.memoryOverhead", "1g")
    .set("spark.driver.memory", "6g")
    .set("spark.driver.maxResultSize", "4g")
    .set("spark.shuffle.service.enabled", "true")
    .set("spark.dynamicAllocation.enabled", "true")
    .set("spark.dynamicAllocation.initialExecutors", "4")
    .set("spark.dynamicAllocation.maxExecutors", "12")
    .set("spark.dynamicAllocation.executorIdleTimeout", "120s")
    .set("spark.dynamicAllocation.cachedExecutorIdleTimeout", "600s")
    .set("spark.dynamicAllocation.shuffleTracking.enabled", "true")
    .set("spark.port.maxRetries", "150")
)

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)


In [ ]:
SOURCE_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"
OUTPUT_DIR = Path.cwd().resolve()
OUTPUT_FILE = OUTPUT_DIR / "model_change_log.xlsx"
MAX_EXCEL_ROWS = 1_048_575

TABLES = {
    "model": f"{SOURCE_DB}.t_model",
    "model_ver": f"{SOURCE_DB}.t_model_ver",
    "model_ver_anlt_dtl": f"{SOURCE_DB}.t_model_ver_anlt_dtl",
    "change_log": f"{SOURCE_DB}.t_ent_param_chg",
}


## Чтение и проверка источников


In [ ]:
def require_columns(df: DataFrame, table_name: str, columns) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise RuntimeError(f"{table_name}: отсутствуют поля: {', '.join(missing)}")


def take_latest(df: DataFrame, keys, order_column: str) -> DataFrame:
    window = Window.partitionBy(*keys).orderBy(F.col(order_column).desc_nulls_last())
    return df.withColumn("__rn", F.row_number().over(window)).filter(F.col("__rn") == 1).drop("__rn")


for table_name in TABLES.values():
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Не найдена таблица: {table_name}")

model_raw = spark.table(TABLES["model"])
model_ver_raw = spark.table(TABLES["model_ver"])
anlt_raw = spark.table(TABLES["model_ver_anlt_dtl"])
change_raw = spark.table(TABLES["change_log"])

required = {
    TABLES["model"]: ["model_sid", "model_name", "model_code", "start_dt"],
    TABLES["model_ver"]: ["model_sid", "model_ver_sid", "model_ver_signfcnt_ctgry_code", "start_dt"],
    TABLES["model_ver_anlt_dtl"]: ["model_ver_sid", "model_ver_stts_name", "model_stts_name", "model_ver_prom_expl_flag", "start_dt"],
    TABLES["change_log"]: ["ent_sid", "ent_type_name", "ent_param_chg_sid", "ent_param_chg_name", "ent_param_chg_prev_val", "ent_param_chg_val", "ent_param_chg_type_code", "ent_param_chg_usr_sid", "ent_param_chg_usr_name", "start_dttm", "end_dttm", "ctl_action", "ctl_datechange"],
}
for table_name, columns in required.items():
    require_columns(spark.table(table_name), table_name, columns)

print("Все таблицы и обязательные поля найдены")


## Все версии моделей в эксплуатации


In [ ]:
model_current = take_latest(model_raw, ["model_sid"], "start_dt").select(
    "model_sid", "model_name", "model_code"
)
model_ver_current = take_latest(model_ver_raw, ["model_sid"], "start_dt").select(
    "model_sid", "model_ver_sid", "model_ver_signfcnt_ctgry_code"
)
version_base = model_ver_current.alias("v").join(
    model_current.alias("m"), F.col("v.model_sid") == F.col("m.model_sid"), "inner"
).select(
    F.col("v.model_ver_sid"),
    F.col("v.model_sid"),
    F.col("m.model_name"),
    F.col("m.model_code"),
    F.col("v.model_ver_signfcnt_ctgry_code"),
)

version_with_status = version_base.alias("v").join(
    anlt_raw.alias("a"), F.col("v.model_ver_sid") == F.col("a.model_ver_sid"), "left"
).select(
    "v.*",
    F.col("a.model_ver_stts_name"),
    F.col("a.model_stts_name"),
    F.col("a.model_ver_prom_expl_flag"),
    F.col("a.start_dt").alias("__anlt_start_dt"),
)
version_context = (
    take_latest(version_with_status, ["model_ver_sid"], "__anlt_start_dt")
    .drop("__anlt_start_dt")
    .filter(F.col("model_ver_prom_expl_flag").cast("boolean") == F.lit(True))
    .select(
        F.col("model_ver_sid").cast("string").alias("model_ver_sid"),
        F.col("model_sid").cast("string").alias("model_sid"),
        "model_name", "model_code", "model_ver_signfcnt_ctgry_code",
        F.col("model_ver_stts_name").alias("current_model_ver_status"),
        F.col("model_stts_name").alias("current_model_status"),
    )
    .cache()
)
operational_count = version_context.count()
if operational_count == 0:
    raise RuntimeError("Не найдено версий моделей с model_ver_prom_expl_flag = true")
print("Версий моделей в эксплуатации:", operational_count)


## Формирование `model_change_log.xlsx`

`previous_value` и `new_value` — состояние параметра до и после конкретного изменения. `current_parameter_value` — последнее значение этого параметра в журнале на момент запуска. Технически удалённые строки (`ctl_action = 'D'`) не включаются.


In [ ]:
effective_changes = change_raw.filter(
    F.col("ctl_action").isNull() | (F.upper(F.trim(F.col("ctl_action"))) != "D")
)
change_rows = (
    version_context.alias("m")
    .join(
        effective_changes.alias("c"),
        (F.col("m.model_ver_sid") == F.col("c.ent_sid").cast("string"))
        & (
            F.upper(F.col("c.ent_param_chg_sid")).startswith("MODEL_VERSION_")
            | F.upper(F.col("c.ent_type_name")).isin("MODEL_VERSION", "MODEL_VER")
        ),
        "inner",
    )
    .select(
        "m.*",
        F.col("c.ent_type_name").alias("entity_type"),
        F.col("c.ent_param_chg_sid").alias("parameter_sid"),
        F.col("c.ent_param_chg_name").alias("parameter_name"),
        F.col("c.ent_param_chg_type_code").alias("change_type_code"),
        F.col("c.start_dttm").alias("change_dttm"),
        F.col("c.end_dttm").alias("period_end_dttm"),
        F.col("c.ctl_datechange").alias("source_change_dttm"),
        F.col("c.ent_param_chg_prev_val").alias("previous_value"),
        F.col("c.ent_param_chg_val").alias("new_value"),
        F.col("c.ent_param_chg_usr_sid").alias("change_user_sid"),
        F.col("c.ent_param_chg_usr_name").alias("change_user_name"),
        F.col("c.ctl_action"),
    )
)

change_desc = Window.partitionBy("model_ver_sid", "parameter_sid").orderBy(
    F.col("change_dttm").desc_nulls_last(),
    F.col("source_change_dttm").desc_nulls_last(),
)
change_asc = Window.partitionBy("model_ver_sid", "parameter_sid").orderBy(
    F.col("change_dttm").asc_nulls_last(),
    F.col("source_change_dttm").asc_nulls_last(),
)
current_value_window = change_desc.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

model_change_log = (
    change_rows
    .withColumn("change_number", F.row_number().over(change_asc))
    .withColumn("is_latest_change", F.row_number().over(change_desc) == 1)
    .withColumn("current_parameter_value", F.first("new_value", ignorenulls=False).over(current_value_window))
    .orderBy("model_ver_sid", "change_dttm", "parameter_sid")
    .cache()
)

change_count = model_change_log.count()
changed_model_count = model_change_log.select("model_ver_sid").distinct().count()
print("Строк изменений:", change_count)
print("Версий моделей с изменениями:", changed_model_count)


In [ ]:
# Pandas/Arrow не поддерживает технические даты вроде 9999-12-31.
# Преобразуем только даты и timestamps в строки ещё на стороне Spark.
excel_df = model_change_log.limit(MAX_EXCEL_ROWS + 1)
temporal_columns = [
    field.name
    for field in excel_df.schema.fields
    if field.dataType.typeName() in {"date", "timestamp", "timestamp_ntz"}
]
for column_name in temporal_columns:
    excel_df = excel_df.withColumn(column_name, F.col(column_name).cast("string"))

result_pdf = excel_df.toPandas()
if len(result_pdf) > MAX_EXCEL_ROWS:
    raise RuntimeError(f"model_change_log.xlsx: {len(result_pdf)} строк — превышен лимит Excel")
result_pdf.to_excel(OUTPUT_FILE, index=False)
print("Сохранён единственный отчёт:", OUTPUT_FILE)

model_change_log.unpersist()
version_context.unpersist()


In [ ]:
# Закрывайте Spark только если после ноутбука сессия больше не нужна.
# spark.stop()
